0. 개념부터 잠깐만 정리
1- 캘리브레이션 하면 뭐가 달라지냐?

렌즈 왜곡(배럴/핀쿠션)이 특히 화면 가장자리에서 꽤 큼

지금 사진도 좌우가 살짝 휘어 있지? (광각 느낌)

그냥 픽셀 수만 비교할 때는 큰 문제 없어 보일 수 있지만,

면적(cm²)까지 계산하거나
cam0 vs cam1 비교, 시계열 비교를 할 때는:

캘리브레이션 안 함 →

모서리 쪽 상추 면적이 실제보다 작거나 크게 나옴

베드 직선이 휘어져서 ROI 기준이 조금씩 틀어짐

캘리브레이션 함 →

베드가 진짜 직선 + 평면으로 펴짐

pixel→mm 변환이 일관됨

→ 정확한 면적·형태 지표까지 가져가려면 캘리브레이션은 하는 게 맞다고 보는 게 좋음.
(특히 너처럼 나중에 논문/모델까지 갈 거면.)

## 1) 기본 설정 셀

In [ ]:
import os
import glob
import cv2
import numpy as np

# ✅ 경로는 연진이가 쓰는 걸로 수정해서 사용
CALIB_DIR = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/calibrated_참고"  # 체커보드 잘 보이는 이미지들만 모아둔 폴더
RAW_DIR   = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본"          # 전체 원본 RGB 폴더
CALIB_OUT_DIR = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/1. Calibrated"  # 보정된 이미지 저장 폴더

os.makedirs(CALIB_OUT_DIR, exist_ok=True)

# ✅ 체커보드 설정
# 격자: 가로 21칸, 세로 8칸 → 내부 코너: 20 x 7
pattern_size = (20, 7)

# 한 칸 실제 크기 (mm)
square_size_x = 20.0      # 가로 20 mm
square_size_y = 37.125    # 세로 37.125 mm


## 2) 캘리브레이션 수행 셀

In [ ]:
def calibrate_camera(calib_dir, pattern_size, square_size_x, square_size_y):
    objpoints = []  # 3D 점 (세계좌표)
    imgpoints = []  # 2D 점 (이미지 좌표)

    # 1) 체커보드 한 장에 대한 3D 좌표 템플릿 생성
    objp = np.zeros((pattern_size[0] * pattern_size[1], 3), np.float32)
    grid = np.mgrid[0:pattern_size[0], 0:pattern_size[1]].T.reshape(-1, 2)
    objp[:, :2] = grid
    objp[:, 0] *= square_size_x   # x 방향 20mm 단위
    objp[:, 1] *= square_size_y   # y 방향 37.125mm 단위

    # 2) 이미지 불러와서 코너 검출
    img_paths = sorted(glob.glob(os.path.join(calib_dir, "*.jpg")))
    print(f"체커보드 캘리브레이션용 이미지 수: {len(img_paths)}")

    gray_shape = None

    for path in img_paths:
        img = cv2.imread(path)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray_shape = gray.shape[::-1]

        ret, corners = cv2.findChessboardCorners(gray, pattern_size, None)

        if ret:
            # 코너 정밀화
            criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
                        30, 0.001)
            corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)

            objpoints.append(objp)
            imgpoints.append(corners2)

            print(f"✅ corners found: {os.path.basename(path)}")
        else:
            print(f"⚠️ corners NOT found: {os.path.basename(path)}")

    # 3) 카메라 보정
    ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
        objpoints, imgpoints, gray_shape, None, None
    )

    # 4) 평균 재투영 오차 출력 (작을수록 좋음, 0.5~1 이하 목표)
    total_error = 0
    for i in range(len(objpoints)):
        imgpoints2, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], mtx, dist)
        error = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2) / len(imgpoints2)
        total_error += error
    mean_error = total_error / len(objpoints)
    print(f"\n📏 mean reprojection error: {mean_error:.4f}")

    return mtx, dist

# 실제 실행
camera_mtx, dist_coeff = calibrate_camera(
    CALIB_DIR,
    pattern_size,
    square_size_x,
    square_size_y
)

# 필요하면 나중에 재사용할 수 있게 저장
np.save("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/1. Calibrated/camera_mtx_cam0.npy", camera_mtx)
np.save("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/1. Calibrated/dist_coeff_cam0.npy", dist_coeff)

print("\n✅ 캘리브레이션 완료")
print("camera matrix:\n", camera_mtx)
print("dist coeffs:\n", dist_coeff.ravel())


체커보드 캘리브레이션용 이미지 수: 21
⚠️ corners NOT found: bed00_20251206_181420_cam0.jpg
⚠️ corners NOT found: bed00_20251206_202158_cam0.jpg
⚠️ corners NOT found: bed01_20251130_060050_cam0.jpg
⚠️ corners NOT found: bed01_20251205_063950_cam1.jpg
⚠️ corners NOT found: bed01_20251205_064340_cam0.jpg
⚠️ corners NOT found: bed01_20251206_073536_cam1.jpg
⚠️ corners NOT found: bed01_20251206_074711_cam1.jpg
⚠️ corners NOT found: bed01_20251206_082746_cam1.jpg
⚠️ corners NOT found: bed02_20251128_214427_cam1.jpg
⚠️ corners NOT found: bed03_20251129_200632_cam1.jpg
⚠️ corners NOT found: bed04_20251204_163010_cam1.jpg
⚠️ corners NOT found: bed04_20251204_163104_cam0.jpg
⚠️ corners NOT found: bed05_20251129_201024_cam1.jpg
⚠️ corners NOT found: bed07_20251130_085318_cam1.jpg
⚠️ corners NOT found: bed09_20251129_201808_cam0.jpg
⚠️ corners NOT found: bed14_20251128_190347_cam1.jpg
⚠️ corners NOT found: bed15_20251128_190545_cam1.jpg
⚠️ corners NOT found: bed24_20251130_085111_cam0.jpg
⚠️ corners NOT found: b

error: OpenCV(4.12.0) /io/opencv/modules/calib3d/src/calibration.cpp:1379: error: (-215:Assertion failed) nimages > 0 in function 'calibrateCameraRO'


## 3) 전체 RGB에 보정 적용해서 저장

In [ ]:
def undistort_and_save_all(raw_dir, out_dir, camera_mtx, dist_coeff):
    os.makedirs(out_dir, exist_ok=True)

    img_paths = sorted(glob.glob(os.path.join(raw_dir, "**", "*.jpg"), recursive=True))
    print(f"보정할 이미지 수: {len(img_paths)}")

    for path in img_paths:
        img = cv2.imread(path)
        if img is None:
            continue

        h, w = img.shape[:2]
        new_camera_mtx, roi = cv2.getOptimalNewCameraMatrix(
            camera_mtx, dist_coeff, (w, h), 1, (w, h)
        )
        undistorted = cv2.undistort(img, camera_mtx, dist_coeff, None, new_camera_mtx)

        # 원래 폴더 구조 유지해서 저장하고 싶으면:
        rel_path = os.path.relpath(path, raw_dir)
        save_path = os.path.join(out_dir, rel_path)
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        cv2.imwrite(save_path, undistorted)

    print("✅ 모든 RGB 보정 이미지 저장 완료.")

# 실행 예시
undistort_and_save_all(RAW_DIR, CALIB_OUT_DIR, camera_mtx, dist_coeff)


## 4. Calibrated 전후 비교하기

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

# ---------------------------------------------------------
# 📌 Before/After 비교 함수
# ---------------------------------------------------------
def show_calibration_before_after(raw_path, camera_mtx, dist_coeff):
    """
    raw_path: 원본 이미지 경로
    """
    img = cv2.imread(raw_path)
    if img is None:
        print("이미지 로딩 실패:", raw_path)
        return

    h, w = img.shape[:2]
    new_camera_mtx, _ = cv2.getOptimalNewCameraMatrix(
        camera_mtx, dist_coeff, (w, h), 1, (w, h)
    )
    corrected = cv2.undistort(img, camera_mtx, dist_coeff, None, new_camera_mtx)

    # BGR → RGB 변환
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    corrected_rgb = cv2.cvtColor(corrected, cv2.COLOR_BGR2RGB)

    # 시각화
    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    plt.title("Before Calibration")
    plt.imshow(img_rgb)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title("After Calibration")
    plt.imshow(corrected_rgb)
    plt.axis("off")

    plt.show()


# ---------------------------------------------------------
# 📌 여러 장 비교 실행 함수
# ---------------------------------------------------------
def compare_samples(sample_paths, camera_mtx, dist_coeff):
    print(f"총 {len(sample_paths)}장의 비교를 실행합니다.\n")
    for path in sample_paths:
        print(f"▶ 비교 중: {os.path.basename(path)}")
        show_calibration_before_after(path, camera_mtx, dist_coeff)
        print("-" * 80)


# ---------------------------------------------------------
# 📌 예시 실행
# ---------------------------------------------------------
# RAW_DIR 안에서 비교할 이미지 3장만 샘플링
sample_list = sorted(
    glob.glob(os.path.join(RAW_DIR, "**", "*.jpg"), recursive=True)
)[:3]

compare_samples(sample_list, camera_mtx, dist_coeff)


### **을 하려고 했지 그런데 해보니까 문제가 많았어
1. 체커보드가 직사각형 2. 크기가 너무작음 3. 반사 등으로 안되어서

보정 대신pixel/mm 스케일링 을 하기로 함.**

In [1]:

"""
체커보드 전체 사각형 기반 pixel/mm 스케일 자동 계산 코드 (v1)

📌 기능
1) calib_dir 폴더 안의 이미지 모두 스캔
2) 체커보드 전체 윤곽(contour) 자동 검출
3) bounding box width_px, height_px 계산
4) 실제 크기 420mm × 297mm 기준으로 px_per_mm_x, px_per_mm_y 계산
5) CSV 저장

📌 이 코드는 연진님 파이프라인에서 STEP 1에 해당합니다.
(ROI, segmentation과 완전히 독립적으로 미리 수행 가능)
"""

import os
import cv2
import numpy as np
import pandas as pd
from glob import glob

# ======================= 설정 ==============================
CALIB_DIR = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251207-251212"  # 체커보드 있는 이미지 폴더
OUTPUT_CSV = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/4. 결과 출력 시각화/1작기(251128-251226)/1파트(251128-251213)/251212_pixel_scale_map.csv"

# 체커보드 전체 실제 크기(mm)
REAL_WIDTH_MM = 420.0
REAL_HEIGHT_MM = 297.0

# ================= 함수 정의 ================================

def extract_checkerboard_box(img):
    """
    체커보드 전체 사각형의 bounding box(width_px, height_px)를 추출.
    실패하면 None 반환.
    """

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # 엣지 검출
    edges = cv2.Canny(blur, 50, 150)

    # 윤곽(contour) 검출
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None

    # 가장 큰 contour 선택 (체커보드가 가장 클 가능성이 높음)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 3000:  # 너무 작은 건 무시
            continue

        rect = cv2.minAreaRect(cnt)
        (cx, cy), (w, h), angle = rect

        # w, h 중 더 긴 쪽을 width, 짧은 쪽을 height으로 정렬
        width_px = max(w, h)
        height_px = min(w, h)

        # 체커보드 크기가 아닌 '바닥 패턴' 등을 잡는 경우 걸러내기
        if width_px < 100 or height_px < 50:
            continue

        return width_px, height_px

    return None


# =================== 메인 루프 =============================

def generate_pixel_scale_csv(calib_dir, output_csv):
    rows = []

    # 하위 폴더까지 탐색하도록 수정
    img_paths = sorted(glob(os.path.join(calib_dir, "**", "*.jpg"), recursive=True))
    print(f"총 {len(img_paths)}장 분석 예정")

    for path in img_paths:
        img = cv2.imread(path)
        if img is None:
            print(f"로드 실패: {path}")
            continue

        result = extract_checkerboard_box(img)
        if result is None:
            print(f"체커보드 박스 검출 실패 → {os.path.basename(path)}")
            continue

        width_px, height_px = result

        px_per_mm_x = width_px / REAL_WIDTH_MM
        px_per_mm_y = height_px / REAL_HEIGHT_MM

        rows.append({
            "image_name": os.path.basename(path),
            "width_px": width_px,
            "height_px": height_px,
            "px_per_mm_x": px_per_mm_x,
            "px_per_mm_y": px_per_mm_y,
        })

        print(f"{os.path.basename(path)} → width_px={width_px:.1f}, height_px={height_px:.1f}")

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"\n📁 스케일 CSV 저장 완료: {output_csv}")
    return df


In [2]:

# =================== 실행 예시 ==============================
df = generate_pixel_scale_csv(CALIB_DIR, OUTPUT_CSV)
display(df.head())

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
체커보드 박스 검출 실패 → bed59_20251208_125637_cam1.jpg
체커보드 박스 검출 실패 → bed59_20251208_160019_cam0.jpg
bed59_20251208_160019_cam1.jpg → width_px=108.8, height_px=84.6
체커보드 박스 검출 실패 → bed59_20251208_160029_cam0.jpg
체커보드 박스 검출 실패 → bed59_20251208_160029_cam1.jpg
체커보드 박스 검출 실패 → bed59_20251208_190413_cam0.jpg
체커보드 박스 검출 실패 → bed59_20251208_190413_cam1.jpg
체커보드 박스 검출 실패 → bed60_20251208_095412_cam0.jpg
체커보드 박스 검출 실패 → bed60_20251208_095412_cam1.jpg
체커보드 박스 검출 실패 → bed60_20251208_125740_cam0.jpg
bed60_20251208_125740_cam1.jpg → width_px=133.5, height_px=85.5
체커보드 박스 검출 실패 → bed60_20251208_125833_cam0.jpg
bed60_20251208_125833_cam1.jpg → width_px=114.9, height_px=95.1
체커보드 박스 검출 실패 → bed60_20251208_160133_cam0.jpg
bed60_20251208_160133_cam1.jpg → width_px=152.0, height_px=86.2
체커보드 박스 검출 실패 → bed60_20251208_160226_cam0.jpg
체커보드 박스 검출 실패 → bed60_20251208_160226_cam1.jpg
체커보드 박스 검출 실패 → bed60_20251208_190621_cam0.jpg
체커보드 박스 검출 실패 → bed60_20251208_190621_cam1.jpg
체커보

,image_name,width_px,height_px,px_per_mm_x,px_per_mm_y
0,bed00_20251207_072520_cam0.jpg,316.378143,287.863373,0.753281,0.969237
1,bed00_20251207_072520_cam1.jpg,148.856995,108.532204,0.354421,0.365428
2,bed00_20251207_112137_cam1.jpg,104.134918,82.201378,0.247940,0.276772
3,bed00_20251207_202534_cam0.jpg,154.070282,144.216827,0.366834,0.485579
4,bed00_20251207_212728_cam0.jpg,141.128479,136.857483,0.336020,0.460800


In [3]:
##1시간 22분 걸림 다음엔 제발 병렬처리해라..